In [ ]:
# %% load needed packages and set seed.
import pandas as pd
import numpy as np
from dataprep.eda import create_report
from datetime import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.feature_selection import mutual_info_classif

from category_encoders import TargetEncoder

from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

In [ ]:
# %% read in data.
train_df = pd.read_csv("Training Data.csv")
scoring_df = pd.read_csv("Scoring Data.csv")

In [ ]:
# %% see a sample of the data.
train_df.head()

noticed that we have current_tier categories but the names are long and really not informative.


also we should check to see if all the tiers are captured so let's get the union of all tiers from both the train and scoring data.

In [ ]:
df = pd.concat([train_df, scoring_df])
current_tier = df["current_tier"].unique().tolist()
current_tier = {
    i: f"Tier {_}"
    for _, i in enumerate(
        df.groupby(["current_tier"])["current_tier"]
        .agg("count")
        .sort_values(ascending=False)
        .index,
        1,
    )
}

train_df["current_tier"] = train_df["current_tier"].map(current_tier)

# let's keep track of columns to exclude from the model
exclude_cols = ["mem_num", "cur_target"]

In [ ]:
# %% simple initial EDA
create_report(train_df).show()

# turn the cur_target variable to a categorical one
train_df["cur_target"] = train_df["cur_target"].astype("category")

 initial thoughts on each feature:
 Balance:
 Balance has a right skew dist. with most of members having a balance between 0 - 35K, with 20% of members having a balance of 0. We did see some outliers with balances of 10.79M and -1.6k. This negative balance, though possible, is very rare and could be fraudulent activity. Seeing as there's only one account with negative balance it's removal will not have a significant impact on model performance.
 depending on the model used log transfomation will be needed to normalize this feature.

 Cardholder (binary):
 The dataset is imbalanced, with more non-cardholders than cardholders.
 could be useful for segmenting customer behavior.

 Current Tier:
 5 tiers, (within the trainig data) with "Tier 1" being the largest group.

 aag_yr1:
 right skewed dist. with 57.1% of members not earning any miles in their first year of enrollment.
 may need to be normalized

 Email Subscriber (binary):
 over 70% of members are signed up for email subs

 Home Airport:
 184 unique values, indicating diverse geographic distribution.
 Seattle (SEA) is the most common airport with 19% of members living closest it.
 Could be useful when looking at regional trends and travel behavior.

 Non-flight Use:
 95.3% members do not redeem miles outside flights.
 could be useful in differentiating between high and low-engagement members.

 Award Use:
 85.6% of members do not redeem miles for flights.

 12 mnth Flight Segments:
 52% of members have not flown in last 12 months.
 roughly 73% of members have 2 or less flight segments within the last 12 months.
 however there are a group of members who very active traveler.

 3 mnth Flight Segments:
 79.3% of members have not flown in the last 3 months. Meaning most members are not frequent flyers.

 12 mnth Non-flight Earned Miles:
 70.3% of members have not earned any non-flight miles within the last 12 months.
 Yet some members earning up to 8.52M miles.
 Could useful for indicate significant engagement differences among users.

 3 mnth Non-flight Earned Miles:
 77.6% of members have not earned any non-flight miles within the last 3 months.

 Lifetime AAG Base Miles:
 25.8% of members have earned zero base miles since enrolling, indicating a lack of engagement specific with flights. Yet there members with large base miles earned. Could useful for member segmentation.

 12 mnth Affinity Spend:
 75.9% of members have zero spending on co-brand cards.
 skewed dist. indicating only a small subset of members actively use co-brand cards.

 3 mnth Affinity Spend:
 79.8% zero values, meaning recent spending is even lower.
 could be useful in predicting member engagement.

 12 mnth Flight Base Miles:
 52% zero values, matching the flight segment distribution.
 Indicates that half of the members do not earn base miles.

 3 mnth Flight Base Miles:
 79.8% zeros, meaning short-term base mile earnings are even lower.
 Aligns with flight segment trends.

 12 mnth Flight bonus Miles:
 52% zero values, similar to base miles.
 Suggests a strong correlation with flight activity.

 3 mnth Flight bonus Miles:
 79.3% zeros, indicating lower short-term bonus mile activity.

 12 mnth vendors Miles Earned:
 84.8% zeros, indicating most members do not earn vendors miles.
 Could be useful for identifying engaged members.

 3 mnth vendors Miles Earned:
 87.1% of members did not earn vendors miles in the last 3 months. could be useful in identifying inactive or low-engagement users.

 12 mnth ptnr_miles_redeemed_12mo
 99.9% of members did not redeem through vendors, indicating that very few users redeem miles through vendors in the last 12 months. This feature is not a useful predictive feature due to its extreme sparsity.

 3 mnth ptnr_miles_redeemed_3mo
 99.96% of the value is zero, meaning almost all members did not redeemed miles through vendors in the past 3 months. This feature is not a useful predictive feature due to its extreme sparsity.

 partner_offer_opt_in:
 partner_offer_opt_in rate is fairly balanced with ~54% of members opting in and 46% opting out. could be useful for identifying engaged members.

 estatement_opt_in:
 64% of members opted in for e-statements. could be useful for identifying engaged members.

 country_cd:
 150 unique countries with most of the members coming from the USA (~97%)

 bg_cur_MV_Count:
 96.7% of members we don't know if they visited the website during the target period it's fair to assume the NaN could imply 0 visits we made. Thus most members did not visit the website during the target period.

 bg_pst_24m_MV_Count:
 84.2% of members likely did not visit the site within the last 24 months.

 bg_pst_24m_Points
 97.9% of members did not purchase points in the last 24 months.

 cur_target:
 the non-target members make up 99.56% of the data points, highly imbalanced dataset and will be considering resampling methods.

 ftb_flag:
 93.57% of members have never purchased our product.

 appu_target:
 most customers did not make any transactions in the target period. This could mean many members are inactive or they did not earn miles through purchases.
 the loyalty program might have a small subset of active users who are making transactions, while most members are inactive


 conclusion:
 The dataset reveals a highly skewed and imbalanced distribution of member activity, engagement, and loyalty behaviors. A significant portion of members exhibit low or no engagement, with many features having high zero percentages, indicating minimal participation in flights, spending, and mile redemptions.


 heatmap:
 appu_target and any other feature that is created during the current target period should be excluded to avoid data leakage. This is because our goal is to predict future promo signups based on prior user behavior, not to use information that only exists because the promotion has already occurred. 
 
 
 appu_target, bg_cur_EST_Create_Date_last and bg_cur_MV_Count are created during the promotion period and are directly influenced by whether a member engaged with the promotion, meaning they capture the outcome rather than the predictors of engagement.

 from the heat map we also see that most features are not correlated cur_target meaning we have to explore interactions between features vs cur_target or non-linear relationship between cur_target the features

 we see strong correlations between features, leading to multicollinearity. This is to be expected as the 12 month specific and 3 month specific features capture the same thing just at different times and see as we get more information from the 12 months than we do 3 months. I will be dropping all 3 months specific features.

 flt_segs_12mo and flt_segs_3mo show strong positive correlation, meaning they capture similar information about flight segments.


 nonflt_earn_12mo and nonflt_earn_3mo also have high correlation, suggesting redundancy.
 affinity_spend_12mo and affinity_spend_3mo follow the same pattern, which may lead to collinearity.


 flt_promo_12mo and flt_promo_3mo have a very high correlation, indicating they measure nearly the same behavior over different timeframes.



 Potential Fix: One of the correlated variables can be dropped or transformed into a new aggregate feature (e.g., total spend over 12mo instead of separate 3mo and 12mo features).

In [ ]:
exclude_cols.extend(
    [
        "flt_segs_3mo",
        "nonflt_earn_3mo",
        "affinity_spend_3mo",
        "flt_base_3mo",
        "flt_promo_3mo",
        "ptnr_miles_earned_3mo",
        "ptnr_miles_redeemed_3mo",
        "appu_target",
        "bg_cur_MV_Count",
        "bg_cur_EST_Create_Date_last",
    ]
)

In [ ]:
##%% checking again the correlation matrix

# Compute correlation matrix
correlation_matrix = (
    train_df[list(set(train_df.columns).difference(set(exclude_cols)))]
    .select_dtypes(include=[np.number])
    .corr()
)


# Heatmap of top correlations
plt.figure(figsize=(12, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap with Reduced Features")
plt.show()

  i see there are still highly correlated features specifically the flight related features:
  flt_segs_12mo and flt_promo_12mo (0.90)
  flt_segs_12mo and flt_base_12mo (0.90)
  flt_promo_12mo and flt_base_12mo (1.00)

  reasonable so as they all capture same behavioural signal flight activities and points based on flight activities
  it might make sense to combine all 3 feature into one by summing them together.
  we could normalize the features (min-max scaling) so that they are all on the same scale and then sum them together.


  nonflt_earn_12mo and affinity_spend_12mo (0.96)
  ptnr_miles_earned_12mo and nonflt_earn_12mo (0.81)
  ptnr_miles_earned_12mo and affinity_spend_12mo (0.83)
  nonflt_earn_12mo and ptnr_miles_earned_12mo (0.81)


  this means that members who earn a lot of non-flight miles tend to have high affinity spending; a reasonable conclusion. again combing the 3 features into one might be the best choice here. we could take an aggregate of them. This helps address the multicollinearity issue.


In [ ]:
# %% creating flight_activity and nonflight_activity feature

# Apply log transformation to handle skewness and avoid divide-by-zero issues
train_df[
    [
        "log_nonflt_earn_12mo",
        "log_affinity_spend_12mo",
        "log_flt_base_12mo",
        "log_flt_promo_12mo",
        "log_flt_segs_12mo",
        "log_ptnr_miles_earned_12mo",
    ]
] = np.log1p(
    train_df[
        [
            "nonflt_earn_12mo",
            "affinity_spend_12mo",
            "flt_base_12mo",
            "flt_promo_12mo",
            "flt_segs_12mo",
            "ptnr_miles_earned_12mo",
        ]
    ]
)

# Normalize using MinMax Scaling
scaler = MinMaxScaler()
train_df[
    [
        "nonflt_earn_12mo_scaled",
        "affinity_spend_12mo_scaled",
        "flt_base_12mo_scaled",
        "flt_promo_12mo_scaled",
        "flt_segs_12mo_scaled",
        "ptnr_miles_earned_12mo_scaled",
    ]
] = scaler.fit_transform(
    train_df[
        [
            "log_nonflt_earn_12mo",
            "log_affinity_spend_12mo",
            "log_flt_base_12mo",
            "log_flt_promo_12mo",
            "log_flt_segs_12mo",
            "log_ptnr_miles_earned_12mo",
        ]
    ]
)


train_df["nonflight_activity"] = (
    train_df["affinity_spend_12mo_scaled"]
    + train_df["nonflt_earn_12mo_scaled"]
    + train_df["ptnr_miles_earned_12mo_scaled"]
)

train_df["flight_activity"] = (
    train_df["flt_base_12mo_scaled"]
    + train_df["flt_promo_12mo_scaled"]
    + train_df["flt_segs_12mo_scaled"]
)

In [ ]:
# %% change date variables to date types the aim here to get an idea for a timeline and age demographic

date_columns = [
    "mp_enrollment_date",
    "most_recent_flt",
    "most_recent_awd",
    "birth_dt_cd",
    "bg_cur_EST_Create_Date_last",
]

train_df[date_columns] = train_df[date_columns].apply(pd.to_datetime)

print(train_df["bg_cur_EST_Create_Date_last"].min())
print(train_df["bg_cur_EST_Create_Date_last"].max())

# it looks like the target period is from Jan to Apr 2024.

In [ ]:
# %% Age
# get age as it's a more meaningful feature than date of birth.
train_df["age"] = dt.now().year - train_df["birth_dt_cd"].dt.year

# exclude birth_dt_cd
exclude_cols.append("birth_dt_cd")
train_df["age"].describe()

x_yes = train_df.loc[(train_df["cur_target"] == 1) & (train_df.age.notna())]["age"]
x_no = train_df.loc[(train_df["cur_target"] == 0) & (train_df.age.notna())]["age"]

fig = ff.create_distplot(
    [x_yes, x_no],
    group_labels=["Yes", "No"],
    show_hist=False,
    show_rug=False,
)
fig.update_layout(
    title=f"Density Plot of Age by Subscription Outcome (Current Target)",
    xaxis_title=f"Age",
    yaxis_title="Density",
)
fig.write_image(f"Density Plot of Age by Current Target Outcome.png")
fig.show()


plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 0], x="age", hue="cur_target", bins=50, kde=True
)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title("Histogram of Age by Subscription Outcome = 0")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 1], x="age", hue="cur_target", bins=50, kde=True
)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title("Histogram of Age by Subscription Outcome = 1")
plt.show()

print(train_df.age.isna().sum() / train_df.shape[0] * 100)

# because the size of missing is so small a simple median imputation should be good enough. Seeing as only 0.18% missing, so the imputation won't introduce significant bias.

train_df["age"].fillna(train_df["age"].median(), inplace=True)

the highest concentration of members are within 30-40 years old.
 we see that people in their mid 30s, early 40s maybe more more likely to subscribe.
 both curves have similar trend, peaking around the same age range. This suggests that age alone may not be a strong discriminator between subscribers and non-subscribers.

 we also have some outliers people with less than 18 or greater than 100. these are likely errors.

In [ ]:
# Identify potential incorrect ages
outlier_ages = train_df[(train_df["age"] < 18) | (train_df["age"] > 100)]
print("Potential outlier ages:")
print(outlier_ages["age"].describe())

In [ ]:
# %% zip code and country
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    return np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))


# Compute Cramér's V for ZIP code and Country
cramers_zip = cramers_v(train_df["zipcode_cd"], train_df["cur_target"])
cramers_country = cramers_v(train_df["country_cd"], train_df["cur_target"])

print(f"Cramér's V for ZIP Code: {cramers_zip}")
print(f"Cramér's V for Country: {cramers_country}")


train_df["y"] = train_df["cur_target"].astype(int)

encoder = TargetEncoder()
train_df["zipcode_encoded"] = encoder.fit_transform(
    train_df["zipcode_cd"], train_df["y"]
)
train_df["country_encoded"] = encoder.fit_transform(
    train_df["country_cd"], train_df["y"]
)


# Compute Mutual Information
mi_zip = mutual_info_classif(
    train_df[["zipcode_encoded"]], train_df["y"], discrete_features=True
)[0]
mi_country = mutual_info_classif(
    train_df[["country_encoded"]], train_df["y"], discrete_features=True
)[0]

print(f"Mutual Information for ZIP Code: {mi_zip:.4f}")
print(f"Mutual Information for Country: {mi_country:.4f}")


exclude_cols.extend(
    ["zipcode_encoded", "zipcode_cd", "country_encoded", "country_cd", "y", "home_apt"]
)

Cramér's V and Mutual Information (MI) results strongly suggest that ZIP Code and Country aren't useful predictors for cur_target.

given the computational cost, dropping ZIP codes and country makes the most practical sense. Given the statistical impact they have cur_target

In [ ]:
# %% Balance

# removing the negative balance
train_df = train_df.loc[train_df.balance >= 0]

plt.figure(figsize=(10, 5))
sns.boxplot(x=train_df["cur_target"], y=train_df["balance"])
plt.yscale("log")  # Apply log scale to handle extreme values
plt.xlabel("Subscription Outcome (cur_target)")
plt.ylabel("Balance (Log Scale)")
plt.title("Box Plot of Balance by Subscription Outcome")
plt.show()


train_df["log_balance"] = np.log1p(train_df["balance"])
plt.figure(figsize=(10, 5))
sns.histplot(train_df, x="log_balance", hue="cur_target", bins=50, kde=True)
plt.xlabel("Balance (Log Scale)")
plt.ylabel("Frequency")
plt.title("Histogram of Balance by Subscription Outcome")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 0],
    x="log_balance",
    hue="cur_target",
    bins=50,
    kde=True,
)
plt.xlabel("Balance (Log Scale)")
plt.ylabel("Frequency")
plt.title("Histogram of Balance by Subscription Outcome = 0")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 1],
    x="log_balance",
    hue="cur_target",
    bins=50,
    kde=True,
)
plt.xlabel("Balance (Log Scale)")
plt.ylabel("Frequency")
plt.title("Histogram of Balance by Subscription Outcome = 1")
plt.show()

#Box Plot:

#The median balance for subscribers (cur_target=1) is noticeably higher than for non-subscribers.
#The distribution of balances for both groups is highly skewed, with extreme outliers; even with the log scaling.

#This means that higher balances may indicate a higher chance of subscribing, but it's not a strong predictor due to large overlap between subscribers and non-subscribers.

hist plot

very skewed dist. for Non-Subscribers, with the largest peak at very low balance values and there's smaller group with a larger balance ranging from ~3k to ~163k. similar to box plot we see that subscribers have higher balance with most of having balance of ~3k to ~163k

we could look into binning the balances into categories like low, medium and high this might help create the more powerful segmentation within our data leading to a stronger predictor. This also reduces the impact of outliers


In [ ]:
# %% binned_balanced

# Define percentiles
low_thresh = train_df["balance"].quantile(0.33)
high_thresh = train_df["balance"].quantile(0.66)

# Apply binning
train_df["balance_binned"] = pd.cut(
    train_df["balance"],
    bins=[-float("inf"), low_thresh, high_thresh, float("inf")],
    labels=["Low", "Medium", "High"],
)


df_melted = (
    train_df.groupby(["cur_target", "balance_binned"])["balance_binned"]
    .agg({"count"})
    .sort_values("count", ascending=False)
    .reset_index()
)
df_melted["percentage"] = (
    df_melted["count"]
    / df_melted.groupby("balance_binned")["count"].transform("sum")
    * 100
)
df_melted = df_melted[df_melted.cur_target == 1]
fig = px.bar(
    df_melted,
    x="balance_binned",
    y="percentage",
    color="balance_binned",
    barmode="group",
    title="Conversion Rate by Balance Binned",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)


fig.write_image(f"Bar Plot of Conversion Rate by Balance Binned.png")
fig.show()

 as we thought the higher balanced members were the most likely to convert followed by the medium balance group and finally the low balance group with smallest conversion rate

 efforts should focused on pushing medium balance members into the high balance group to improve conversions.

 though higher tiers increases the likelihood of responding, however,  absolute response rates remain very low.

In [ ]:
# %% current_tier
df_melted = (
    train_df.groupby(["cur_target", "current_tier"])["current_tier"]
    .agg({"count"})
    .sort_values("count", ascending=False)
    .reset_index()
)
df_melted["percentage"] = (
    df_melted["count"]
    / df_melted.groupby(["current_tier"])["count"].transform("sum")
    * 100
)

df_melted = df_melted[df_melted.cur_target == 1]
fig = px.bar(
    df_melted,
    x="current_tier",
    y="percentage",
    color="current_tier",
    title="Conversion Rate by Current Tier",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)
fig.write_image(f"Bar Plot of Conversion Rate by Current Tier.png")
fig.show()

#Most customers (~95%) belong to Tier 1, as mentioned before, by default it will have the largest number of members subscribing.

#let's specifically look at conversion rates by tiers
#tier 5 has the highest conversion rate and tier 1 as the lowest we see a general trend that the higher the tier the more likely the member is to convert. likely more loyal and engaged members who are more likely to engage in new promos offered.

though higher tiers increases the likelihood of responding, however,  absolute response rates remain very low.

In [ ]:
# Set Seaborn style
sns.set(style="whitegrid")


plt.figure(figsize=(10, 5))
sns.boxplot(x="current_tier", y="affinity_spend_12mo", data=train_df)
plt.yscale("log")  # Log scale to account for skewed spending data
plt.title("Spending Behavior Across Tiers (Log Scale)")
plt.xlabel("Current Tier")
plt.ylabel("Affinity Spend (Log Scale)")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(x="current_tier", y="flt_segs_12mo", data=train_df)
plt.yscale("log")  # Log scale due to skewness
plt.title("Flight Activity Across Tiers (Log Scale)")
plt.xlabel("Current Tier")
plt.ylabel("Flight Segments (Last 12 Months)")
plt.show()

# Set the figure size for better readability
plt.figure(figsize=(10, 5))

# 1. Box plot for bg_pst_24m_MV_Count across tiers
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="current_tier", y="bg_pst_24m_MV_Count", showfliers=False)
plt.yscale("log")  # Apply log scale due to skewness
plt.title("Website Visits in the Past 24 Months by Tier (Log Scale)")
plt.xlabel("Current Tier")
plt.ylabel("Website Visits (Log Scale)")
plt.show()

# 2. Box plot for bg_pst_24m_Points across tiers
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="current_tier", y="bg_pst_24m_Points", showfliers=False)
plt.yscale("log")  # Apply log scale due to skewness
plt.title("Points Balance in the Past 24 Months by Tier (Log Scale)")
plt.xlabel("Current Tier")
plt.ylabel("Points Balance (Log Scale)")
plt.show()

# 3. Stacked bar chart for ftb_flag across tiers
ftb_tier_dist = (
    train_df.groupby("current_tier")["ftb_flag"].value_counts(normalize=True).unstack()
)

plt.figure(figsize=(10, 5))
ftb_tier_dist.plot(kind="bar", stacked=True, colormap="coolwarm", figsize=(10, 5))
plt.title("First-Time Buyer Distribution Across Tiers")
plt.xlabel("Current Tier")
plt.ylabel("Proportion of Members")
plt.legend(title="First-Time Buyer (ftb_flag)")
plt.show()

# Check If Lower Tiers Are Newer Customers
train_df["enrollment_year"] = train_df["mp_enrollment_date"].dt.year

plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="current_tier", y="enrollment_year")
plt.title("Enrollment Year by Tier")
plt.xlabel("Current Tier")
plt.ylabel("Year of Enrollment")
plt.xticks(rotation=45)
plt.show()

 we see that overall the higher the tier the greater the spending meaning more engaged more likely to convert

#if the member has high flight activity they tend to be in higher tiers, again implying more engaged more likely to convert.

#Higher tiers have a lower proportion of first-time buyers, suggesting that members in these tiers have likely made multiple purchases in the past.
#Tier 1 has the highest proportion of first-time buyers, which makes sense as these members are likely newer or have lower engagement.

#Tiers 4 and 5 have a noticeably lower proportion of first-time buyers, indicating that members in these tiers are more experienced and more likely to be repeat purchasers.

#Points balance increases with higher membership tiers. This aligns with the idea that higher-tier members are more engaged and have accumulated more points over time.

#website visit is fairly consist across the tiers but we see that higher tiers have slightly more site visits, though this difference is minimal.


#Members in Tier 1 tend to be recently enrolled, though we do have some outliers with enrollment dating back severl decades.
#Higher tiers generally have members who have been enrolled for a longer time. meaning longevity in the program is correlated with higher-tier status.

In [ ]:
# %% Total Miles Earned 1st Year

train_df["log_aag_yr1"] = np.log1p(train_df["aag_yr1"])
plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 0],
    x="log_aag_yr1",
    hue="cur_target",
    bins=50,
    kde=True,
)
plt.xlabel("Total Miles Earned 1st Year (Log Scale)")
plt.ylabel("Frequency")
plt.title("Histogram of Earned Miles by Subscription Outcome = 0")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(
    train_df[train_df.cur_target == 1],
    x="log_aag_yr1",
    hue="cur_target",
    bins=50,
    kde=True,
)
plt.xlabel("Total Miles Earned 1st Year (Log Scale)")
plt.ylabel("Frequency")
plt.title("Histogram of Earned Miles by Subscription Outcome = 1")
plt.show()

# hist.
# both plot show that most members have earned little to no miles during their first year.
#
# there is smaller group of members who earned 3k to ~22k miles

In [ ]:
# %% aag_yr1_binned
# Calculate quantiles (excluding zero earners)
q1 = train_df.loc[train_df["aag_yr1"] > 0, "aag_yr1"].quantile(0.25)
q3 = train_df.loc[train_df["aag_yr1"] > 0, "aag_yr1"].quantile(0.75)

# Define updated bin edges
bins = [-np.inf, 0, q1, q3, np.inf]
labels = ["Non-Earners", "Low-Earners", "Medium-Earners", "High-Earners"]

# Apply binning
train_df["aag_yr1_binned"] = pd.cut(train_df["aag_yr1"], bins=bins, labels=labels)

df_melted = (
    train_df.groupby(["cur_target", "aag_yr1_binned"])["aag_yr1_binned"]
    .agg({"count"})
    .sort_values("count", ascending=False)
    .reset_index()
)
df_melted["percentage"] = (
    df_melted["count"]
    / df_melted.groupby(["aag_yr1_binned"])["count"].transform("sum")
    * 100
)

df_melted = df_melted[df_melted.cur_target == 1]
fig = px.bar(
    df_melted,
    x="aag_yr1_binned",
    y="percentage",
    color="aag_yr1_binned",
    title="Conversion Rate by Total Miles Earned 1st Yr",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)
fig.write_image(f"Bar Plot of Conversion Rate by Total Miles Earned 1st Yr.png")
fig.show()

#we see that those who are high earners the most likely to convert, implying early engagement strongly correlates with promotional responsiveness. and as the earning level gets lower so to the conversion rate. the promo should be aimed at high and medium earners,

#though higher miles earned in 1st yr increases the likelihood of responding, however,  absolute response rates remain very low.

In [ ]:
# %% cardholder
df_melted = (
    train_df.groupby(["cur_target", "cardholder"])["cardholder"]
    .agg({"count"})
    .sort_values("count", ascending=False)
    .reset_index()
)
df_melted["percentage"] = (
    df_melted["count"] / df_melted.groupby("cardholder")["count"].transform("sum") * 100
)
df_melted = df_melted[df_melted.cur_target == 1]
fig = px.bar(
    df_melted,
    x="cardholder",
    y="percentage",
    color="cardholder",
    barmode="group",
    title="Conversion Rate by Cardholder",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)

fig.write_image(f"Bar Plot of Conversion Rate by Cardholder.png")
fig.show()

#Most people in the dataset are not cardholders.
#~77% are non-cardholders, while ~23% are cardholders.

#Amongst non-cardholders, only 0.26% responded to the promotion.
#while cardholders, 0.99% responded—almost 4x higher engagement than non-cardholders.
#Despite higher engagement rates, most responders are still non-cardholders

#Business & Modeling Implications
#Being a cardholder increases the likelihood of responding (though absolute response rates remain very low).
#The marketing team may want to target cardholders more aggressively since they are more engaged.

In [ ]:
# Boxplot for Affinity Spend in the last 12 months
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="affinity_spend_12mo")
plt.yscale("log")
plt.xlabel("Cardholder Status")
plt.ylabel("Affinity Spend (Last 12 Months, Log Scale)")
plt.title("Affinity Spend in the Last 12 Months by Cardholder Status")
plt.show()


# Boxplot for Flight Miles Earned in the last 12 months
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="flt_base_12mo")
plt.yscale("log")
plt.xlabel("Cardholder Status")
plt.ylabel("Flight Miles Earned (12 Months, Log Scale)")
plt.title("Flight Miles Earned in the Last 12 Months by Cardholder Status")
plt.show()

# Boxplot for Flight Miles Redeemed
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="awd_use")
plt.yscale("log")
plt.xlabel("Cardholder Status")
plt.ylabel("Flight Miles Redeemed (Log Scale)")
plt.title("Flight Miles Redeemed by Cardholder Status")
plt.show()

# Boxplot for Non-Flight Miles Earned in the last 12 months
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="nonflt_earn_12mo")
plt.yscale("log")
plt.xlabel("Cardholder Status")
plt.ylabel("Non-Flight Miles Earned (12 Months, Log Scale)")
plt.title("Non-Flight Miles Earned in the Last 12 Months by Cardholder Status")
plt.show()

# Boxplot for Non-Flight Miles Redeemed
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="nonflt_use")
plt.yscale("log")
plt.xlabel("Cardholder Status")
plt.ylabel("Non-Flight Miles Redeemed (Log Scale)")
plt.title("Non-Flight Miles Redeemed by Cardholder Status")
plt.show()


# Convert enrollment date to year
train_df["enrollment_year"] = train_df["mp_enrollment_date"].dt.year

# Boxplot for Enrollment Year by Cardholder Status
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="enrollment_year")
plt.xlabel("Cardholder Status")
plt.ylabel("Year of Enrollment")
plt.title("Enrollment Year by Cardholder Status")
plt.show()

# Set Seaborn style
sns.set_style("whitegrid")

# Log-transform for better visualization
train_df["log_bg_pst_24m_Points"] = np.log1p(train_df["bg_pst_24m_Points"])
train_df["log_flt_segs_12mo"] = np.log1p(train_df["flt_segs_12mo"])


###Points Purchased in the Last 24 Months by Cardholder Status
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=train_df, x="cardholder", y="log_bg_pst_24m_Points", palette="coolwarm"
)
plt.xlabel("Cardholder Status")
plt.ylabel("Points Purchased (Log Scale)")
plt.title("Points Purchased in the Last 24 Months by Cardholder Status")
plt.show()

###Flight Segments in the Last 12 Months by Cardholder Status
plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="cardholder", y="log_flt_segs_12mo", palette="coolwarm")
plt.xlabel("Cardholder Status")
plt.ylabel("Flight Segments (Last 12 Months, Log Scale)")
plt.title("Flight Segments in the Last 12 Months by Cardholder Status")
plt.show()

#cardholder tend to spend more than non-cardholders

#cardholders tend to earn and redeem more flight miles than non-cardholders.

#card holders earn more non-flight miles than non-cardholders

#non-flight redemption in both group is fairly similar where most members don't redeem non-flight miles

#most cardholders have been enrolled in the program much longer, while non-cardholders tend to be new to program.

#cardholders tend purchase slight more points than non-cardholders

#cardholder fly more than non-cardholders

#Cardholder status is a strong predictor of travel behavior, spending, and engagement in the loyalty program. However, both groups have extreme outliers regardless of what feature it's compared with.


In [ ]:
# %% email_subscriber
df_melted = (
    train_df.groupby(["cur_target", "email_subscriber"])["cur_target"]
    .agg({"count"})
    .reset_index()
)

df_melted["percentage"] = (
    df_melted["count"]
    / df_melted.groupby(["email_subscriber"])["count"].transform("sum")
    * 100
)
# Count plot of current tier by cardholder status
fig = px.bar(
    df_melted,
    x="email_subscriber",
    y="count",
    color="cur_target",
    title="Distribution of Email Subscription Across Target Outcome",
    labels={"count": "Count", "email_subscriber": "Email Subscription"},
    barmode="group",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)
fig.write_image(f"Bar Plot of Email Subscription by Target Outcome.png")
fig.show()

#Email Subscription Majority:

#The majority of both groups (target = 0 and target = 1) are subscribed to emails.
#Over 99.5% of email subscribers fall into the non-subscription category (cur_target = 0).
#Similarly, 99.71% of non-subscribers also fall into cur_target = 0.

#A very small group of members who are subscribed to emails have converted (cur_target = 1).
#Though the difference in conversion rates between email subscribers and non-subscribers is quite large, however, absolute response rates remain very low.

In [ ]:
# %% partner opt in

df_melted = (
    train_df.groupby(["cur_target", "partner_offer_opt_in"])["cur_target"]
    .agg({"count"})
    .reset_index()
)

df_melted["percentage"] = (
    df_melted["count"]
    / df_melted.groupby(["partner_offer_opt_in"])["count"].transform("sum")
    * 100
)
# Count plot of current tier by cardholder status
fig = px.bar(
    df_melted,
    x="partner_offer_opt_in",
    y="count",
    color="cur_target",
    title="Distribution of Partner Offer Opt In Across Target",
    labels={"count": "Count", "email_subscriber": "Partner Opt"},
    barmode="group",
    text=df_melted["percentage"].apply(lambda x: f"{x:.2f}%"),
)
fig.write_image(f"Bar Plot of Partner Offer Opt In Across Target Outcome.png")
fig.show()

# we see that those who opt in have a higher conversion rate however, the absolute response rates remain very low.

In [ ]:
# %% mp_enrollment_date

# it may be more useful to have mp_enrollment_date represented as tenure_yrs, a similar logic that was applied to date of birth presented by age.

train_df["tenure_yrs"] = dt.now().year - train_df["enrollment_year"]

reference_date = pd.Timestamp("2025-02-20", tz="UTC")
train_df["days_since_last_flt"] = (reference_date - train_df["most_recent_flt"]).dt.days
train_df["days_since_last_awd"] = (reference_date - train_df["most_recent_awd"]).dt.days


fig = px.histogram(
    train_df[train_df.cur_target == 0],
    "tenure_yrs",
    title="Program Tenure Vs Target Outcome = 0",
)
fig.write_image(f"Bar Plot of Program Tenure Vs Target Outcome = 0.png")
fig.show()
fig = px.histogram(
    train_df[train_df.cur_target == 1],
    "tenure_yrs",
    title="Program Tenure Vs Target Outcome = 1",
)
fig.write_image(f"Bar Plot of Program Tenure Vs Target Outcome = 1.png")
fig.show()

fig = px.histogram(
    train_df[train_df.cur_target == 0],
    "days_since_last_flt",
    title="Days Since Last Flt Vs Target Outcome = 0",
)
fig.write_image(f"Bar Plot of Days Since Last Flt Vs Target Outcome = 0.png")
fig.show()
fig = px.histogram(
    train_df[train_df.cur_target == 1],
    "days_since_last_flt",
    title="Days Since Last Flt Vs Target Outcome = 1",
)
fig.write_image(f"Bar Plot of Days Since Last Flt Vs Target Outcome = 1.png")
fig.show()

fig = px.histogram(
    train_df[train_df.cur_target == 0],
    "days_since_last_awd",
    title="Days Since Last Awd Vs Target Outcome = 0",
)
fig.write_image(f"Bar Plot of Days Since Last AwdVs Target Outcome = 0.png")
fig.show()
fig = px.histogram(
    train_df[train_df.cur_target == 1],
    "days_since_last_awd",
    title="Days Since Last Awd Vs Target Outcome = 1",
)
fig.write_image(f"Bar Plot of Days Since Last Awd Vs Target Outcome = 1.png")
fig.show()

#shows what we already concluded, a large number of non-subscribers are new members. Where as subscribers tend be have in program for over 8 yrs.
#%%####EDA Conclusion######

#The dataset is highly skewed and imbalanced, with most members exhibiting low engagement across flights, spending, and mile redemptions. Feature engineering focused on handling missing values, normalizing skewed distributions (e.g., log transformations, MinMax scaling), and reducing multicollinearity by aggregating highly correlated features (e.g., flight activity and spending). Key insights revealed that higher-tier members, long-tenured members, and engaged spenders were more likely to subscribe to promotions, while first-time buyers and low-mile earners had lower conversion rates. Given the class imbalance, resampling techniques may be necessary.
